In [ ]:
import heapq
import sys
from collections import defaultdict, deque
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

In [ ]:
def parse_file(filepath):
    """Parse a file into a list of (int, str, str) tuples."""
    result = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            result.append((int(parts[0]), parts[1], parts[2]))
    return result

In [ ]:
def merge_sorted_files(file_data_list):
    """
    Merge N sorted arrays into one sorted array using a min-heap.
    """
    # Min-heap: (int_value, file_index, entry_index, str1, str2)
    heap = []
    
    # Initialize heap with first element from each non-empty file
    for j in range(len(file_data_list)):
        entry = file_data_list[j][0]
        # Push: (int_value, j, entry_index, str1, str2)
        heapq.heappush(heap, (entry[0], j, 0, entry[1], entry[2]))
    
    result = []
    while heap:
        int_val, j, idx, str1, str2 = heapq.heappop(heap)
        # Append with source as second element: (int, "h{j}", str, str)
        result.append((int_val, f"h{j+1}", str1, str2))
        
        # If there are more entries in this file, push the next one
        next_idx = idx + 1
        if next_idx < len(file_data_list[j]):
            next_entry = file_data_list[j][next_idx]
            heapq.heappush(heap, (next_entry[0], j, next_idx, next_entry[1], next_entry[2]))
    
    return result

In [ ]:
def read_seed(n, seed):
    file_data_list = []
    for i in range(n):
        file_name = f"../results/{n}/{seed}/evnt_h{i+1}.log"
        file_data = parse_file(file_name)
        if len(file_data) > 0:
            file_data_list.append(file_data)
        else:
            print(f"  [warn] read_seed: file '{file_name}' is empty")

    return merge_sorted_files(file_data_list)

In [ ]:
class UUIDGraph:
    def __init__(self):
        self.subject_uuid = {}          # subject -> their current uuid
        self.adj = defaultdict(set)     # uuid -> set of related uuids
        self.active_uuids = set()       # all currently active uuids
        self.pending_edges = defaultdict(set)  # target_uuid -> set of src uuids waiting on it

    def _activate_uuid(self, uuid):
        """Mark uuid as active and flush any pending edges targeting it."""
        self.active_uuids.add(uuid)
        if uuid in self.pending_edges:
            for src in self.pending_edges.pop(uuid):
                if src in self.active_uuids:  # src may have been X'd in the meantime
                    self._add_edge(src, uuid)

    def _add_edge(self, u, v):
        if u != v:
            self.adj[u].add(v)
            self.adj[v].add(u)

    def _remove_edge(self, u, v):
        self.adj[u].discard(v)
        self.adj[v].discard(u)

    def _remove_uuid(self, uuid):
        """Remove uuid and all its edges. Also scrub it from pending_edges."""
        self.active_uuids.discard(uuid)
        # Remove all adjacency edges
        for neighbor in list(self.adj[uuid]):
            self.adj[neighbor].discard(uuid)
        if uuid in self.adj:
            del self.adj[uuid]
        # Remove as a pending source (it was waiting on some target)
        for targets in self.pending_edges.values():
            targets.discard(uuid)
        # Remove as a pending target (others were waiting on it — discard those intentions)
        if uuid in self.pending_edges:
            del self.pending_edges[uuid]

    def process(self, event):
        ts, subject, etype, uuid = event

        if etype == 'E':
            self.subject_uuid[subject] = uuid
            self._activate_uuid(uuid)

        elif etype == 'J':
            my_uuid = self.subject_uuid.get(subject)
            if my_uuid is None:
                print(f"  [warn] J: subject '{subject}' has no uuid", file=sys.stderr)
                return
            if uuid in self.active_uuids:
                self._add_edge(my_uuid, uuid)
            else:
                # Buffer: wait until target uuid becomes active
                self.pending_edges[uuid].add(my_uuid)

        elif etype == 'L':
            my_uuid = self.subject_uuid.get(subject)
            if my_uuid is None:
                print(f"  [warn] L: subject '{subject}' has no uuid", file=sys.stderr)
                return
            # Cancel pending edge if it hasn't been flushed yet
            if uuid in self.pending_edges:
                self.pending_edges[uuid].discard(my_uuid)
            self._remove_edge(my_uuid, uuid)

        elif etype == 'X':
            my_uuid = self.subject_uuid.get(subject)
            if my_uuid is None:
                print(f"  [warn] X: subject '{subject}' has no uuid", file=sys.stderr)
                return
            if my_uuid != uuid:
                print(f"  [warn] X: subject '{subject}' holds {my_uuid}, not {uuid}", file=sys.stderr)
            self._remove_uuid(my_uuid)
            del self.subject_uuid[subject]

    def stats(self):
        """Return (total_edges, component_sizes, pending_edge_count)."""
        total_edges = sum(len(v) for v in self.adj.values()) // 2
        pending = sum(len(v) for v in self.pending_edges.values())

        visited = set()
        components = []
        for start in self.active_uuids:
            if start in visited:
                continue
            size = 0
            queue = deque([start])
            visited.add(start)
            while queue:
                node = queue.popleft()
                size += 1
                for neighbor in self.adj[node]:
                    if neighbor not in visited:
                        visited.add(neighbor)
                        queue.append(neighbor)
            components.append(size)

        return total_edges, sorted(components, reverse=True), pending

    def print_stats(self):
        edges, components, pending = self.stats()
        print(f"edges={edges}  pending={pending}  components={components}")

In [ ]:
def write_file(path, n, seed):
    data = read_seed(n, seed)
    with open(path, "w") as f:
        graph = UUIDGraph()
        data_len = len(data)
        count = 0
        for event in tqdm(data):
            graph.process(event)
            edges, components, _ = graph.stats()
            expected = 0
            for c in components:
                expected += c * (c - 1) // 2
            if expected == 0:
                f.write(f'{event[0]} N/A\n')
            else:
                f.write(f'{event[0]} {edges/expected}\n')
            count += 1

In [ ]:
N_PEERS = 101
N_ITERS = 1

In [ ]:
for i in range(N_ITERS):
    write_file(f'../processed_data/reachability/{N_PEERS}/{i}.txt', N_PEERS, i)

In [ ]:
def plot_one_seed(seed):
    with open(f'../processed_data/reachability/{N_PEERS}/{i}.txt', 'r') as f:
        time_init = 0
        time_points = []
        reachabilities = []
        
        for line in f:
            parts = line.strip().split()
            if parts[1] == 'N/A':
                continue
            
            time = int(parts[0])
            reachability = float(parts[1])
            if time_init == 0:
                time_init = time
            time_points.append((time - time_init) / 1000)
            reachabilities.append(reachability * 100)
            
        plt.plot(time_points, reachabilities, color='blue', alpha=0.3)

for i in range(N_ITERS):
    plot_one_seed(i)

# plt.ylim(90, 100.1)
# plt.xlim(118, 140)
plt.xlabel('Time (s)')
plt.ylabel('Reachability (%)')
plt.grid(True)
plt.savefig('plot.jpg', dpi=300, bbox_inches='tight')
plt.show()